In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib

In [ ]:
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")
acceptances = pd.read_excel("../../data/cleaned/acceptances/acceptances.xlsx")

In [3]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = acceptances.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,POINT THE STAR,IN THE STARS
1,SPARKLING THEA,SPARKLING DEW
2,PRICELESSGIRL,PRICELESS GOLD
3,ROYAL ECLAIR,ROYAL CLASSIC
4,CAPTAIN,CATALINA
...,...,...
201,STARLIGHTER,STARLIGHT DANCER
202,MAGNETAR,MAGENTA
203,QUEEN CARLA,QUEEN CAROLINE
204,GOLDEN HEART,GOLDEN HAMMER


In [10]:
import re
import pandas as pd

unique_names = acceptances.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [ ]:
# =========================
# 2. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['venue'] = df['venue'].str.strip().str.upper()
    df['horse_name'] = df['horse_name'].str.strip().str.upper()
    return df

runners = normalize(runners)
acceptances = normalize(acceptances)

# =========================
# 3. BUILD TRUE MAPPING (FROM RUNNERS)
# =========================
runner_map = runners[
    ['meet_date', 'venue', 'horse_name', 'race_no']
].drop_duplicates()

# =========================
# 4. MERGE (IGNORE ACCEPTANCE RACE_NO)
# =========================
merged = acceptances.merge(
    runner_map,
    on=['meet_date', 'venue', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_acc', '_true')
)

# =========================
# 5. KEEP ONLY VALID RUNNERS
# =========================
clean_acceptances = merged[merged['_merge'] == 'both'].copy()

# overwrite race_no using runners (ground truth)
clean_acceptances['race_no'] = clean_acceptances['race_no_true']

# drop helper columns
clean_acceptances = clean_acceptances.drop(
    columns=['_merge', 'race_no_acc', 'race_no_true']
)

# =========================
# 6. EDGE CASES (DROPPED)
# =========================
edge_cases = merged[merged['_merge'] == 'left_only'].copy()

edge_case_list = edge_cases[
    ['meet_date', 'venue', 'race_no_acc', 'horse_name']
].drop_duplicates()

print("Dropped (non-runners):", len(edge_case_list))
display(edge_case_list.head())

# =========================
# 7. FINAL CLEAN
# =========================
clean_acceptances = clean_acceptances.sort_values(
    by=['meet_date', 'venue', 'race_no']
).reset_index(drop=True)

# =========================
# 8. SAVE
# =========================
clean_acceptances.to_excel("../../data/cleaned/acceptances_cleaned/acceptances.xlsx", index=False)

Dropped (non-runners): 2321


,meet_date,venue,race_no_acc,horse_name
307,2018-01-18,MUMBAI,7,ODESSA
308,2018-01-18,MUMBAI,7,GLORIOUS EYES
916,2018-02-23,MUMBAI,7,INCENTIO
917,2018-02-23,MUMBAI,7,FLASHY WINGS
918,2018-02-23,MUMBAI,7,ETERNALINSPIRATION
